# 05 — Classical: Exact Binary Model (MILP, PuLP + CBC)

---

### What this notebook does

Solves the same problem **exactly**, on all **472** focus orders at once, with an off-the-shelf
MILP solver. Where the greedy commits to one order at a time and never revisits it, the solver
weighs every combination of moves together and returns the provable optimum.

That makes this the **accuracy ceiling** for the optimality-gap discussion: the distance between
it and the greedy is what the heuristic leaves on the table.

### Three things the brief asks for

1. **An off-the-shelf MILP solver** — PuLP with CBC, which ships with it.
2. **An accuracy ceiling** — only true if the greedy's answer is *inside* the solver's feasible
   set. Section 6 verifies `MILP ≥ greedy` at every instance size rather than assuming it.
3. **Wall-clock time, as input to Section 8** — recorded here as a *curve* across five instance
   sizes, not a single number, because that is what a scaling discussion needs.

### What it does not do

No cleaning, no rule definitions. The objective and C1–C7 arrive from `dom_model`, exactly as in
`03` and `04`. What is added is only a second way of *searching*. The thing being searched over is
identical, which is what makes the gap meaningful.

## 1. Setup

In [1]:
import os, sys, time, subprocess
import numpy as np
import pandas as pd

try:
    import pulp                                  # already installed?
except ImportError:                              # Colab: install it once
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pulp"], check=True)
    import pulp

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 80)

In [2]:
import os, glob, zipfile

# Colab keeps nothing between runtimes, so the shared folder goes on Drive when we can.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/DOM"
except Exception:
    WORK = os.path.abspath("dom_work")

CLEAN   = f"{WORK}/clean"      # notebook 02 writes here
RESULTS = f"{WORK}/results"    # notebooks 03, 04, 05 write here
os.makedirs(CLEAN, exist_ok=True)
os.makedirs(RESULTS, exist_ok=True)
print("work folder:", WORK)

Mounted at /content/drive
work folder: /content/drive/MyDrive/DOM


In [3]:
# The model lives in exactly one file, written by 02_data_cleaning.
# Data, constraints C1-C7 and the objective all arrive from here.
sys.path.insert(0, WORK)
os.environ["DOM_CLEAN"] = CLEAN
from dom_model import *

print("orders:", len(HEAD), "| focus:", len(FOCUS), "| clean:", len(CLEAN_ORDERS))
print("DCs:", DCS, "| days:", NT)

orders: 1109 | focus: 472 | clean: 637
DCs: [5083, 5385, 5410, 5420, 5490, 5620, 5641, 5773] | days: 32


In [4]:
POOL_A, DEF = stage_A()                # same starting point as 03 and 04
FOCUS_SET   = list(FOCUS)              # the full 472 — no subsetting
print("focus orders in the model:", len(FOCUS_SET))

TIME_LIMIT  = 900     # seconds per solve
MIP_GAP     = 0.0     # 0 = solve to proven optimality
SOLVER_MSG  = 0       # 1 to watch CBC's log
SWEEP_SIZES = [50, 100, 200, 300, len(FOCUS_SET)]   # must match SWEEP_SIZES in 04_greedy

focus orders in the model: 472


## 2. The model

### 2.1 Decision variables

| variable | type | meaning |
|---|---|---|
| `x[g,d]` | binary | order `g` is served from DC `d` |
| `q[g,d,s]` | continuous ≥ 0 | cases of SKU `s` filled for order `g` at DC `d` |
| `z[g]` | binary | order `g` reached its fill-rate threshold, so no penalty |
| `pen[g]` | continuous ≥ 0 | the penalty charged on order `g` |

Quantities are variables rather than fixed per option. That matters: with fixed quantities the
model cannot express a partial fill, and it comes out **below** the greedy — which would make the
optimality gap meaningless.

### 2.2 Objective

Maximise `Σ price·q − Σ pen − Σ ship·x`, which is `Revenue − Penalty − Shipping cost`, the same
expression `objective()` evaluates.

### 2.3 The seven constraints

| # | as written in the doc | as written in the code below |
|---|---|---|
| C1 | one order, one DC | `lpSum(x[g,d] for d) == 1` |
| C2 | never fill more than ordered | `q[g,d,s] <= dem * x[g,d]` |
| C3 | stock there, covering 5 days | `lpSum(q drawing on day u) <= free stock + handed back` |
| C4 | 5 points **and** 100 cases | `lpSum(q[g,d,s] for s) >= need * x[g,d]` |
| C5 | picks fit the daily limit | `lpSum(q/cpp) - lpSum(lines*x) <= pallet room` |
| C6 | one dock per order | `lpSum(x[g,d] at that DC/day) <= free slots` |
| C7 | penalty below threshold | `pen + ppc*lpSum(price*q) >= full*(1 - z)`, `lpSum(q) >= threshold*z` |

C2, C4 and C7 are the only ones needing care. C2 both caps the fill and switches it off when the
DC is not chosen. C4 is a straight linearisation of the gate. C7 is the standard big-M trick for
"charge this only if the threshold was missed" — maximising the objective pushes `pen` down onto
its lower bound, which is exactly what `penalty_of()` computes.

### 2.4 One deliberate choice about C3

`avail()` checks stock as a rolling minimum over the next 5 days, but `take()` removes it from
*every* later day. Those are not the same rule. The MILP follows `avail()` — each draw is
constrained over its own cover window — so both methods obey an identical C3 and the gap measures
search quality, not two different inventory models.

`strict_ledger=True` swaps in the stricter day-by-day version. Section 8 runs it as a sensitivity,
because the difference is itself a finding.

### 2.5 Candidate options

For each order: its default DC, plus every DC that stocks all its SKUs and can still hit the
requested delivery date. These become the `x` variables. The C4 gate is *not* used to filter here —
it is a constraint in the model, so the solver sees the same option set the greedy did.

In [5]:
def candidate_options(subset):
    """order -> [(dc, ship-day index, is_default)]"""
    cand = {}
    for g in subset:
        base = DEF[g]
        opts = [(base["dc"], base["t"], True)]                 # staying put is always an option
        for d in DCS:
            if d == base["dc"]:
                continue
            if not all(s in SKU_AT_DC[d] for s, _, _, _ in LINES[g]):
                continue                                       # DC lacks a SKU
            p, err = revised_pgi(g, d)                         # date still works?
            if err:
                continue
            opts.append((d, DIX[p], False))
        cand[g] = opts
    return cand

CAND = candidate_options(FOCUS_SET)
print(f"orders: {len(CAND)} | options: {sum(len(v) for v in CAND.values())} "
      f"({sum(len(v)-1 for v in CAND.values())} alternates, {len(CAND)} defaults)")

orders: 472 | options: 2295 (1823 alternates, 472 defaults)


### 2.6 Build the model

Each constraint is written the way it appears in the table above, so the code and the formulation
can be checked against each other line by line.

The stock rows need one piece of bookkeeping. `POOL_A` is what is left *after* Stage A, so it
already has the focus orders' own draws taken out of it. Since the model may move those orders,
each one hands its Stage-A draw back — `GIVE` — and the right-hand side becomes
`free stock + what was handed back`.

In [6]:
def build_model(subset, cand, strict_ledger=False):
    W = CFG["FORWARD_COVER_DAYS"]

    # stock the subset hands back = its own Stage-A draws, from its ship day onward
    GIVE = {}
    for g in subset:
        r = DEF[g]
        for s, qty, _, _ in r["fills"]:
            if qty > 0:
                GIVE.setdefault((r["dc"], s), np.zeros(NT))[r["t"]:] += qty

    # dock room: every default order holds a slot (clamped at 0, as in the greedy),
    # then the focus orders hand their own slot back so it can be reassigned
    dock_av = DOCK0.copy()
    for g, r in DEF.items():
        i, j = DCIX[r["dc"]], r["t"]
        if DOCK_HAS[i, j]:
            dock_av[i, j] -= CFG["DOCKS_PER_ORDER"]
    dock_av = np.maximum(0.0, dock_av)
    for g in subset:
        r = DEF[g]
        i, j = DCIX[r["dc"]], r["t"]
        if DOCK_HAS[i, j]:
            dock_av[i, j] += CFG["DOCKS_PER_ORDER"]

    prob = pulp.LpProblem("DOM", pulp.LpMaximize)

    # ---- variables ----
    x   = {(g, d): pulp.LpVariable(f"x_{g}_{d}", cat="Binary")
           for g in subset for d, _, _ in cand[g]}
    q   = {(g, d, s): pulp.LpVariable(f"q_{g}_{d}_{s}", lowBound=0, upBound=dem)
           for g in subset for d, _, _ in cand[g] for s, dem, _, _ in LINES[g] if dem > 0}
    z   = {g: pulp.LpVariable(f"z_{g}", cat="Binary")
           for g in subset if HEAD[g]["ppc"] > 0}
    pen = {g: pulp.LpVariable(f"pen_{g}", lowBound=0)
           for g in subset if HEAD[g]["ppc"] > 0}

    # ---- objective: revenue - penalty - shipping ----
    prob += (pulp.lpSum(price * q[g, d, s]
                        for g in subset for d, _, _ in cand[g]
                        for s, dem, price, _ in LINES[g] if dem > 0)
             - pulp.lpSum(pen.values())
             - pulp.lpSum(SHIP.get((d, HEAD[g]["zipc"]), 0.0) * x[g, d]
                          for g in subset for d, _, _ in cand[g]))

    # ---- C1  one order, one DC ----
    for g in subset:
        prob += pulp.lpSum(x[g, d] for d, _, _ in cand[g]) == 1

    # ---- C2  fill <= demand, and only at the chosen DC ----
    for g in subset:
        for d, _, _ in cand[g]:
            for s, dem, _, _ in LINES[g]:
                if dem > 0:
                    prob += q[g, d, s] <= dem * x[g, d]

    # ---- C3  stock, over each draw's cover window ----
    draws = {}
    for g in subset:
        for d, t, isdef in cand[g]:
            w = 0 if isdef else W                       # stage A uses today only
            for s, dem, _, _ in LINES[g]:
                if dem > 0:
                    draws.setdefault((d, s), []).append((q[g, d, s], t, w))
    stock_rows = 0
    for (d, s), items in draws.items():
        arr, giv = POOL_A.get((d, s)), GIVE.get((d, s))
        days = set()
        for _, t, w in items:
            days |= set(range(t, min(NT, t + w + 1)))
        for u in sorted(days):
            if strict_ledger:      # a draw removes stock from every later day
                active = [v for v, t, w in items if t <= u]
            else:                  # doc rule: each draw is checked over its own window
                active = [v for v, t, w in items if t <= u <= t + w]
            if not active:
                continue
            rhs = (max(0.0, arr[u]) if arr is not None else 0.0) + (giv[u] if giv is not None else 0.0)
            prob += pulp.lpSum(active) <= rhs
            stock_rows += 1

    # ---- C4  the divert gate ----
    for g in subset:
        h, base = HEAD[g], DEF[g]
        need = base["filled"] + max(CFG["MIN_FILL_LIFT_PP"] * h["ordered_cases"],
                                    CFG["MIN_CASE_LIFT"])
        for d, _, isdef in cand[g]:
            if isdef:
                continue
            prob += pulp.lpSum(q[g, d, s] for s, dem, _, _ in LINES[g] if dem > 0) \
                    >= need * x[g, d]

    # ---- C5  pallet picks, only where the cell could actually overflow ----
    pe, pick_rows = {}, 0
    for g in subset:
        for d, t, isdef in cand[g]:
            if not isdef:
                pe.setdefault((d, t), []).append(g)
    for (d, t), gs in pe.items():
        i = DCIX[d]
        hi_pp = sum(sum(dem / cpp for _, dem, _, cpp in LINES[g]) for g in gs)
        if hi_pp <= PP0[i, t]:
            continue                                   # cannot overflow -> no row needed
        prob += (pulp.lpSum(q[g, d, s] / cpp
                            for g in gs for s, dem, _, cpp in LINES[g] if dem > 0)
                 - pulp.lpSum(len(LINES[g]) * x[g, d] for g in gs)) <= float(PP0[i, t])
        pick_rows += 1

    # ---- C6  one dock slot per order ----
    de, dock_rows = {}, 0
    for g in subset:
        for d, t, _ in cand[g]:
            if DOCK_HAS[DCIX[d], t]:
                de.setdefault((d, t), []).append(x[g, d])
    for (d, t), vs in de.items():
        cap = float(dock_av[DCIX[d], t])
        if len(vs) > cap:
            prob += pulp.lpSum(vs) <= cap
            dock_rows += 1

    # ---- C7  penalty below the threshold ----
    for g in subset:
        h = HEAD[g]
        if h["ppc"] <= 0:
            continue
        full = sum(dem * price for _, dem, price, _ in LINES[g]) * h["ppc"]      # big M
        prob += (pen[g] + h["ppc"] * pulp.lpSum(price * q[g, d, s]
                                                for d, _, _ in cand[g]
                                                for s, dem, price, _ in LINES[g] if dem > 0)
                 >= full * (1 - z[g]))
        prob += pulp.lpSum(q[g, d, s] for d, _, _ in cand[g]
                           for s, dem, _, _ in LINES[g] if dem > 0) >= h["threshold_cases"] * z[g]

    info = dict(nvars=len(x) + len(q) + len(z) + len(pen), nbin=len(x) + len(z),
                nrows=len(prob.constraints), stock_rows=stock_rows,
                dock_rows=dock_rows, pick_rows=pick_rows)
    return prob, x, q, z, pen, info


t0 = time.time()
prob, X, Q, Z, PEN, info = build_model(FOCUS_SET, CAND)
print(f"built in {time.time()-t0:.1f}s")
print(f"variables : {info['nvars']:,}  ({info['nbin']:,} binary)")
print(f"rows      : {info['nrows']:,}  (stock {info['stock_rows']:,} · "
      f"dock {info['dock_rows']} · picks {info['pick_rows']})")

built in 4.0s
variables : 88,342  (2,594 binary)
rows      : 148,663  (stock 60,264 · dock 24 · picks 33)


## 3. Solve

CBC ships with PuLP, so there is nothing else to install. `gapRel = 0` asks for a proven optimum
rather than a good-enough one; `timeLimit` stops it if that proves too expensive.

The wall clock around is `prob.solve()`.

In [7]:
def solve(prob, time_limit=TIME_LIMIT, mip_gap=MIP_GAP, msg=SOLVER_MSG):
    t0 = time.time()
    prob.solve(pulp.PULP_CBC_CMD(msg=msg, timeLimit=time_limit, gapRel=mip_gap))
    wall = time.time() - t0
    return wall, pulp.LpStatus[prob.status], pulp.value(prob.objective)


WALL, STATUS, SOLVER_OBJ = solve(prob)
print(f"status   : {STATUS}")
print(f"wall time: {WALL:.2f} s")
print(f"objective: {SOLVER_OBJ:,.2f}")

status   : Optimal
wall time: 63.72 s
objective: 46,295,828.48


## 4. Decode the answer

Turn the solver's variables back into the same per-order record the other notebooks produce, then
**re-score it with `objective()` and `penalty_of()`**. If the two agree, the model and the scorer
are the same thing and the comparison in `06` is sound.

In [8]:
def decode(subset, cand, x, q):
    res = {}
    for g in subset:
        h = HEAD[g]
        for d, t, isdef in cand[g]:
            if (x[g, d].value() or 0.0) <= 0.5:
                continue                                       # not the chosen DC
            by = {s: ((q[g, d, s].value() or 0.0) if (g, d, s) in q else 0.0)
                  for s, dem, _, _ in LINES[g]}
            fills = [(s, by.get(s, 0.0), price, cpp) for s, dem, price, cpp in LINES[g]]
            tot   = sum(by.values())
            rev   = sum(by[s] * price for s, dem, price, _ in LINES[g])
            cp, pp = picks(fills)
            res[g] = dict(dc=d, t=t, fills=fills, by=by, filled=tot, revenue=rev,
                          cp=cp, pp=pp,
                          pen=penalty_of(g, by, tot),                # scored by dom_model
                          ship=SHIP.get((d, h["zipc"]), 0.0),
                          cof=tot / h["ordered_cases"] if h["ordered_cases"] else 0.0,
                          diverted=not isdef, chosen_dc=d, lift=tot - DEF[g]["filled"])
            break
    return res


MILP = decode(FOCUS_SET, CAND, X, Q)
for g in CLEAN_ORDERS:                    # the 637 clean orders never move
    MILP[g] = dict(DEF[g], diverted=False, chosen_dc=DEF[g]["dc"], lift=0.0)

rescored = sum(objective(MILP[g]) for g in FOCUS_SET)
print(f"solver objective   : {SOLVER_OBJ:,.2f}")
print(f"re-scored objective: {rescored:,.2f}")
print(f"difference         : {SOLVER_OBJ - rescored:,.2f}   (solver tolerance)")
print(f"orders moved       : {sum(1 for g in FOCUS_SET if MILP[g]['diverted'])}")

solver objective   : 46,295,828.48
re-scored objective: 46,295,828.48
difference         : 0.00   (solver tolerance)
orders moved       : 66


## 5. Feasibility check

The decoded answer is pushed back through the same checks the greedy applies one order at a time.

One point about tolerance. C4 enters the model as `lpSum(q) >= need * x`, and a MILP solver
satisfies that to its own feasibility tolerance — it will happily stop a fraction of a case short.
Re-checking with exact arithmetic then flags orders that are, in every practical sense, fine. So
the audit reports **how far** each check is missed, and counts a breach only above a thousandth of
a case. The same tolerance produces the small gap between the two objectives above.

In [9]:
TOL = 1e-3        # cases. Anything under this is solver rounding, not a broken rule.

bad, c4_slack = [], []
for g in FOCUS_SET:
    r, h = MILP[g], HEAD[g]
    dem_of = dict((a, b) for a, b, _, _ in LINES[g])
    for s, qty, _, _ in r["fills"]:                                  # C2
        if qty - dem_of[s] > TOL:
            bad.append((g, "C2 over-fill", qty - dem_of[s]))
    if r["diverted"]:
        if not all(s in SKU_AT_DC[r["dc"]] for s, _, _, _ in LINES[g]):
            bad.append((g, "SKU not carried", 1.0))
        need = max(CFG["MIN_FILL_LIFT_PP"] * h["ordered_cases"],      # C4
                   CFG["MIN_CASE_LIFT"])
        c4_slack.append(need - r["lift"])
        if need - r["lift"] > TOL:
            bad.append((g, "C4 gate", need - r["lift"]))
        p, err = revised_pgi(g, r["dc"])                              # ship date
        if err or DIX[p] != r["t"]:
            bad.append((g, "ship date", 1.0))

print(f"orders checked : {len(FOCUS_SET)}   (moved: {len(c4_slack)})")
print(f"worst C4 miss  : {max(c4_slack):.6f} cases   <- solver feasibility tolerance")
print(f"breaches above tolerance ({TOL} cases): {len(bad)}")
if bad:
    print(pd.DataFrame(bad, columns=["order","check","amount"])
            .groupby("check")["amount"].agg(["count","max"]).to_string())
else:
    print("all clear — no rule broken beyond solver rounding")

orders checked : 472   (moved: 66)
worst C4 miss  : 0.000000 cases   <- solver feasibility tolerance
breaches above tolerance (0.001 cases): 0
all clear — no rule broken beyond solver rounding


## 6. Timed sweep — the accuracy ceiling, verified at every size

This is the heart of the notebook.

The exact model is solved at five instance sizes, and at each one the wall clock, the solver
status and the greedy's answer **on that same subset** are recorded. The greedy numbers come from
`metrics_greedy_by_size.csv`, which `04` produces by genuinely re-running the greedy on each
subset — not by filtering its full-472 result, which would give a wrong comparison.

Two things to read off it:

* **`ceiling_ok`** — is `MILP ≥ greedy` at this size? Every row must be `True`, otherwise "accuracy
  ceiling" is not a claim this notebook can make.
* **`wall_s`** — where the exact method stops being tractable, which is the empirical answer to
  what "tractable subset" means on this data.

In [10]:
BYSIZE = pd.read_csv(f"{RESULTS}/metrics_greedy_by_size.csv").set_index("orders")

sweep = []
for n in SWEEP_SIZES:
    if n == len(FOCUS_SET):                       # reuse the model already solved above
        w, st, ob, nb, nr = WALL, STATUS, rescored, info["nbin"], info["nrows"]
    else:
        sub = sorted(FOCUS, key=lambda g: -HEAD[g]["revenue"])[:n]
        c   = candidate_options(sub)
        p_, x_, q_, z_, pn_, i_ = build_model(sub, c)
        w, st, _ = solve(p_)
        r_  = decode(sub, c, x_, q_)
        ob, nb, nr = sum(objective(r_[g]) for g in sub), i_["nbin"], i_["nrows"]
    gre = float(BYSIZE.loc[n, "objective"])       # greedy, run on this same subset
    sweep.append(dict(orders=n, binaries=nb, rows=nr, wall_s=round(w, 2), status=st,
                      milp=ob, greedy=gre,
                      gap_pct=(ob - gre) / abs(ob) * 100,
                      ceiling_ok=ob >= gre - 1e-6))

SW = pd.DataFrame(sweep)
print(SW.to_string(index=False,
                   formatters={"milp": "{:,.0f}".format, "greedy": "{:,.0f}".format,
                               "gap_pct": "{:+.3f}".format}))
print()
print("ceiling holds at every size:", bool(SW.ceiling_ok.all()))

 orders  binaries   rows  wall_s  status       milp     greedy gap_pct  ceiling_ok
     50       227  35830    1.41 Optimal 12,444,505 12,340,615  +0.835        True
    100       458  64976    6.43 Optimal 20,866,440 20,432,810  +2.078        True
    200       920  99153   18.45 Optimal 33,830,660 32,777,117  +3.114        True
    300      1472 123972   35.13 Optimal 41,904,450 40,595,549  +3.124        True
    472      2594 148663   63.72 Optimal 46,295,828 44,810,604  +3.208        True

ceiling holds at every size: True


### 6.1 What the sweep shows

Two readings, and the second is the more interesting one.

In [11]:
print("wall-clock growth")
for a, b in zip(SW.itertuples(), SW.iloc[1:].itertuples()):
    print(f"  {a.orders:3d} -> {b.orders:3d} orders : binaries x{b.binaries/a.binaries:.1f}, "
          f"time x{(b.wall_s/max(a.wall_s,1e-6)):.1f}")

print()
print("optimality gap of the greedy, by instance size")
for r in SW.itertuples():
    print(f"  {r.orders:3d} orders: {r.gap_pct:+.3f}%")
print()
print("-> the gap widens with size: the more orders there are to coordinate, the more a")
print("   one-pass heuristic loses by never revisiting an earlier decision.")

wall-clock growth
   50 -> 100 orders : binaries x2.0, time x4.6
  100 -> 200 orders : binaries x2.0, time x2.9
  200 -> 300 orders : binaries x1.6, time x1.9
  300 -> 472 orders : binaries x1.8, time x1.8

optimality gap of the greedy, by instance size
   50 orders: +0.835%
  100 orders: +2.078%
  200 orders: +3.114%
  300 orders: +3.124%
  472 orders: +3.208%

-> the gap widens with size: the more orders there are to coordinate, the more a
   one-pass heuristic loses by never revisiting an earlier decision.


## 7. Optimality gap at full size

The headline numbers for the report, on all 472 focus orders.

In [12]:
gm = pd.read_csv(f"{RESULTS}/metrics_greedy.csv")
bm = pd.read_csv(f"{RESULTS}/metrics_baseline.csv")
g_best = gm.loc[gm.objective_focus.idxmax()]
b_obj  = float(bm.objective_focus.iloc[0])

print(f"{'Baseline 1':32s} {b_obj:>15,.0f}")
print(f"{g_best.scenario:32s} {g_best.objective_focus:>15,.0f}   "
      f"moves {int(g_best.orders_diverted):3d}   {g_best.runtime_s:.2f}s")
print(f"{'Classical (MILP, exact)':32s} {rescored:>15,.0f}   "
      f"moves {sum(1 for g in FOCUS_SET if MILP[g]['diverted']):3d}   {WALL:.2f}s")
print("-" * 70)
print(f"optimality gap of the greedy : {(rescored - g_best.objective_focus)/abs(rescored)*100:.3f}%")
print(f"MILP gain over Baseline 1    : ${rescored - b_obj:,.0f}")
print(f"greedy captured              : {(g_best.objective_focus - b_obj)/(rescored - b_obj)*100:.1f}% of it")

Baseline 1                            44,365,994
Greedy 2A · by order value            44,810,604   moves  41   0.40s
Classical (MILP, exact)               46,295,828   moves  66   63.72s
----------------------------------------------------------------------
optimality gap of the greedy : 3.208%
MILP gain over Baseline 1    : $1,929,835
greedy captured              : 23.0% of it


## 8. Sensitivity — the stricter inventory ledger

Section 2.4 explained the choice. Here it is measured: the same model with C3 enforced day by day
across the whole horizon rather than over each draw's 5-day window.

The stricter version is the more conservative inventory story, and it is worth knowing how much of
the headline gain survives it. It is also a **harder model**, so it gets its own cap. If it stops
at the limit, the objective shown is the best solution *found* — a lower bound, not a proven
optimum. The `proven` column says which.

In [13]:
STRICT_LIMIT = 120        # this variant is harder; cap it and report what it reached

prob2, X2, Q2, Z2, PEN2, info2 = build_model(FOCUS_SET, CAND, strict_ledger=True)
wall2, st2, _ = solve(prob2, time_limit=STRICT_LIMIT)
M2 = decode(FOCUS_SET, CAND, X2, Q2)
strict_obj = sum(objective(M2[g]) for g in FOCUS_SET)
proven2 = (st2 == "Optimal")

print(pd.DataFrame([
    dict(model="cover window (doc rule, default)", rows=info["nrows"],
         objective=round(rescored, 0),
         moves=sum(1 for g in FOCUS_SET if MILP[g]["diverted"]),
         wall_s=round(WALL, 2), proven=(STATUS == "Optimal")),
    dict(model="strict day-by-day ledger", rows=info2["nrows"],
         objective=round(strict_obj, 0),
         moves=sum(1 for g in FOCUS_SET if M2[g]["diverted"]),
         wall_s=round(wall2, 2), proven=proven2),
]).to_string(index=False))

print()
print("strict-ledger solver status:", st2)
if not proven2:
    print("-> stopped at the cap, so this row is the best solution FOUND: a lower bound on that")
    print("   variant's optimum. It is not the proven optimum and must not be read as one.")

print()
print(f"for scale:  greedy 2A = {g_best.objective_focus:>13,.0f}")
print(f"            strict    = {strict_obj:>13,.0f}   ({strict_obj - g_best.objective_focus:+,.0f} vs greedy)")
print(f"            default   = {rescored:>13,.0f}   ({rescored - strict_obj:+,.0f} vs strict)")
print()
print("A stricter rule can only lower the optimum, so default >= strict is expected.")

                           model   rows  objective  moves  wall_s  proven
cover window (doc rule, default) 148663 46295828.0     66   63.72    True
        strict day-by-day ledger 148663 44910045.0     65  151.68    True

strict-ledger solver status: Optimal

for scale:  greedy 2A =    44,810,604
            strict    =    44,910,045   (+99,441 vs greedy)
            default   =    46,295,828   (+1,385,784 vs strict)

A stricter rule can only lower the optimum, so default >= strict is expected.


## 9. Save

Identical filenames and columns to the previous version, so `06_comparison` needs no change. The
sweep is written alongside them as the scaling evidence.

In [14]:
mm = metrics(MILP, "Classical · MILP (exact)", round(WALL, 2))
mm["solver"]      = "CBC (PuLP)"
mm["status"]      = STATUS
mm["n_variables"] = info["nvars"]
mm["n_binaries"]  = info["nbin"]
mm["n_rows"]      = info["nrows"]
mm["optimality_gap_of_greedy_pct"] = (rescored - g_best.objective_focus) / abs(rescored) * 100

to_frame(MILP, "classical").to_csv(f"{RESULTS}/results_classical.csv", index=False)
pd.DataFrame([mm]).to_csv(f"{RESULTS}/metrics_classical.csv", index=False)
SW.to_csv(f"{RESULTS}/classical_scaling.csv", index=False)
print("wrote results_classical.csv, metrics_classical.csv, classical_scaling.csv")

wrote results_classical.csv, metrics_classical.csv, classical_scaling.csv


## 10. What this tells us

* **The ceiling is verified, not assumed.** `MILP ≥ greedy` holds at every instance size in the
  sweep, so the gap is a real distance-to-optimality rather than an artefact of two models that
  quietly differ.
* **The greedy is close in percentage terms and far in useful terms.** Most of the money sits in
  orders that were never in trouble. Measured against the *gain over Baseline 1* — the part anyone
  can influence — the shortfall is much larger than the headline percentage suggests.
* **The gap widens with instance size.** On small instances the greedy is nearly optimal; the more
  orders there are to coordinate, the more it loses by never revisiting a decision. That is the
  argument for the exact model, and it is the one the sweep supports directly.
* **The solver moves considerably more orders.** Planning all 472 together lets it chain moves:
  vacating one DC frees stock that lets a second order divert in. A one-pass heuristic cannot see
  that.
* **Runtime is what to watch as this grows.** The binary count — orders × candidate DCs — drives
  branch-and-bound effort, and the sweep shows how it climbs. Batching focus orders by DC or by
  ship week is the obvious way to keep instances tractable.
* **The remaining limit is the data, not the search.** No forecast file means no stock is reserved
  at the receiving DC, so both methods are more willing to divert than the 2024 POC was.